# Recife Monitora - Entendimento dos Dados

**Projetos 5 - Gestão de Tecnologia da Informação - CESAR School**

>> **Base sintética:** os valores utilizados neste notebook são simulados e não representam resultados reais da Secretaria de Saúde do Recife.

## Objetivo

Esta etapa tem como objetivo compreender a estrutura e a qualidade inicial dos dados antes de qualquer tratamento ou análise exploratória.

Serão avaliados:
- estrutura e granularidade das tabelas
- dimensões dos conjuntos de dados
- tipos das variáveis
- valores ausentes
- unicidade das chaves
- domínios das variáveis
- integridade referencial entre as tabelas
- aderência às regras de negócio definidas no dicionário de dados

As inconsistências encontradas serão inicialmente identificadas e documentadas e as decisões de tratamento serão realizadas posteriormente na etapa de preparação dos dados.

In [1]:
import pandas as pd
import numpy as np

In [2]:
from pathlib import Path 
Path.cwd()

WindowsPath('c:/Users/marce/OneDrive/Documentos/Marcela/vital-data-lab/notebooks')

In [3]:
CAMINHO_DADOS = Path("../data/raw")
CAMINHO_DADOS

list(CAMINHO_DADOS.glob("*.csv"))

[WindowsPath('../data/raw/eixo3_indicadores.csv'),
 WindowsPath('../data/raw/equipes.csv'),
 WindowsPath('../data/raw/nps_atributos.csv'),
 WindowsPath('../data/raw/planos_acao.csv'),
 WindowsPath('../data/raw/resultados_ciclos.csv')]

In [4]:
equipes = pd.read_csv(CAMINHO_DADOS / "equipes.csv")
resultados = pd.read_csv(CAMINHO_DADOS / "resultados_ciclos.csv")
eixo3 = pd.read_csv(CAMINHO_DADOS / "eixo3_indicadores.csv")
nps = pd.read_csv(CAMINHO_DADOS / "nps_atributos.csv")
planos = pd.read_csv(CAMINHO_DADOS / "planos_acao.csv")

print(f"equipes: {equipes.shape}")
print(f"resultados: {resultados.shape}")
print(f"eixo3: {eixo3.shape}")
print(f"nps: {nps.shape}")
print(f"planos: {planos.shape}")

equipes: (132, 9)
resultados: (1392, 13)
eixo3: (13294, 7)
nps: (6815, 6)
planos: (420, 9)


## Visão geral dos conjuntos de dados

A base é composta por cinco tabelas com diferentes níveis de granularidade.Nesta etapa inicial, foram verificadas as dimensões de cada conjunto de dados, sem aplicação de tratamentos ou alterações nos registros.

| Tabela | Registros | Variáveis | Grão esperado |
|---|---:|---:|---|
| equipes | 132 | 9 | Uma equipe |
| resultados_ciclos | 1.392 | 13 | Uma equipe em um ciclo |
| eixo3_indicadores | 13.294 | 7 | Um indicador de uma equipe em um ciclo |
| nps_atributos | 6.815 | 6 | Um atributo de uma equipe em um ciclo |
| planos_acao | 420 | 9 | Um plano de ação |

Os volumes apresentados representam os dados brutos recebidos. A aderência ao grão esperado, a unicidade das chaves e a integridade entre as tabelas serão verificadas nas etapas seguintes.

## 1. Tabela equipes.csv

In [5]:
equipes.head(10)


,codigo_cnes,codigo_ine,nome_unidade,nome_equipe,distrito_sanitario,tipo_equipe,grupo,populacao_adscrita,n_profissionais
0,2437842,6000000000,USF V-842,Equipe 001,Distrito Sanitário V,eSF,2,2455.0,15
1,2451211,6000000001,USF VIII-211,Equipe 002,Distrito Sanitário VIII,eSB,3,3606.0,21
2,2423763,6000000002,USF VII-763,Equipe 003,Distrito Sanitário VII,eSF,1,2920.0,16
3,2424542,6000000003,USF VII-542,Equipe 004,Distrito Sanitário VII,eMulti,4,3581.0,27
4,2450984,6000000004,USF VII-984,Equipe 005,Distrito Sanitário VII,eSF,2,NaN,33
5,2477497,6000000005,USF V-497,Equipe 006,Distrito Sanitário V,esf,1,10088.0,46
6,2483603,6000000006,USF VI-603,Equipe 007,DISTRITO SANITÁRIO VI,eSF,2,5342.0,24
7,2451211,6000000007,USF VIII-211,Equipe 008,Distrito Sanitário VIII,eMulti,1,1352.0,10
8,2459204,6000000008,USF V-204,Equipe 009,Distrito Sanitário V,eSF,1,8110.0,35
9,2409295,6000000009,USF I-295,Equipe 010,Distrito Sanitário I,eMulti,4,3787.0,21


In [6]:
equipes.info()

<class 'pandas.DataFrame'>
RangeIndex: 132 entries, 0 to 131
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   codigo_cnes         132 non-null    int64  
 1   codigo_ine          132 non-null    int64  
 2   nome_unidade        132 non-null    str    
 3   nome_equipe         132 non-null    str    
 4   distrito_sanitario  132 non-null    str    
 5   tipo_equipe         132 non-null    str    
 6   grupo               132 non-null    int64  
 7   populacao_adscrita  121 non-null    float64
 8   n_profissionais     132 non-null    int64  
dtypes: float64(1), int64(4), str(4)
memory usage: 9.4 KB


In [7]:
print(f"código_ine únicos: {equipes["codigo_ine"].nunique()}")
print(f"código_ine duplicados: {equipes["codigo_ine"].duplicated().sum()}")

código_ine únicos: 132
código_ine duplicados: 0


### Validação do grão e da chave primária

- A tabela `equipes.csv` possui 132 registros e 132 valores distintos de `codigo_ine`, sem ocorrência de duplicidades nessa coluna.
- Como o dicionário de dados estabelece que cada linha deve representar uma única equipe e define `codigo_ine` como chave primária, os resultados indicam que a granularidade esperada está sendo respeitada.

**Resultado:** não foram identificadas duplicidades na chave primária `codigo_ine`.

In [8]:
equipes.isna().sum()

codigo_cnes            0
codigo_ine             0
nome_unidade           0
nome_equipe            0
distrito_sanitario     0
tipo_equipe            0
grupo                  0
populacao_adscrita    11
n_profissionais        0
dtype: int64

In [9]:
(equipes.isna().mean() * 100).round(2)

codigo_cnes           0.00
codigo_ine            0.00
nome_unidade          0.00
nome_equipe           0.00
distrito_sanitario    0.00
tipo_equipe           0.00
grupo                 0.00
populacao_adscrita    8.33
n_profissionais       0.00
dtype: float64

### Verificação de valores ausentes

- A análise de completude identificou valores ausentes apenas na variável `populacao_adscrita`.
- Foram encontrados **11 registros ausentes entre as 132 equipes**, correspondendo a aproximadamente **8,33%** dos registros da tabela.
- As demais variáveis não apresentaram valores nulos.

In [10]:
print(f"Quantidade de valores distintos de distritos sanitários: {equipes["distrito_sanitario"].nunique()}")
print(f"\nDistribuição dos valores:\n{equipes["distrito_sanitario"].value_counts()}")

Quantidade de valores distintos de distritos sanitários: 19

Distribuição dos valores:
distrito_sanitario
Distrito Sanitário VI      18
Distrito Sanitário V       15
Distrito Sanitário VIII    15
Distrito Sanitário VII     14
Distrito Sanitário I       13
Distrito Sanitário III     13
Distrito Sanitário IV      13
Distrito Sanitário II      12
ds v                        4
DISTRITO SANITÁRIO V        3
Distrito Sanitario IV       3
DISTRITO SANITÁRIO VI       2
ds ii                       1
DISTRITO SANITÁRIO I        1
DS III                      1
DISTRITO SANITÁRIO VII      1
DS VII                      1
Distrito Sanitario V        1
DS II                       1
Name: count, dtype: int64


In [11]:
print(f"Quantidade de valores distintos de tipos de equipe: {equipes["tipo_equipe"].nunique()}")
print(f"\nDistrbuição de valores:\n{equipes["tipo_equipe"].value_counts()}")

Quantidade de valores distintos de tipos de equipe: 12

Distrbuição de valores:
tipo_equipe
eSF        71
eSB        21
eMulti     14
e-SF        6
ESF         6
esf         3
 eSF        3
e-SB        2
ESB         2
esb         2
EMULTI      1
e-Multi     1
Name: count, dtype: int64


In [12]:
print(f"Quantidade de valores distintos de grupos: {equipes["grupo"].nunique()} ")
print(f"\nDistribuição de valores:\n{equipes["grupo"].value_counts()}")

Quantidade de valores distintos de grupos: 4 

Distribuição de valores:
grupo
2    40
1    38
3    37
4    17
Name: count, dtype: int64


### Validação dos domínios categóricos

Foram avaliadas as variáveis categóricas `distrito_sanitario`, `tipo_equipe` e `grupo`, comparando os valores encontrados na base com os domínios definidos no dicionário de dados.

#### Distrito sanitário

O dicionário prevê **8 distritos sanitários (I a VIII)**. Entretanto, foram encontradas **19 representações distintas** na base.
As divergências estão relacionadas principalmente a diferenças de capitalização, abreviações e acentuação, como:

- `Distrito Sanitário V`
- `DISTRITO SANITARIO V`
- `Distrito Sanitario V`
- `ds V`

Esses valores representam conceitualmente o mesmo distrito, mas são interpretados como categorias diferentes pelo Pandas.

#### Tipo de equipe

O domínio esperado contém apenas três categorias: `eSF`, `eSB` e `eMulti`, mas foram encontradas **12 representações distintas**.
Foram observadas variações de capitalização, presença de hífen e espaços adicionais, como `eSF`, `ESF`, `esf`, `e-SF` e ` eSF`.Essas variações provocam a criação artificial de categorias distintas para o mesmo tipo de equipe.

#### Grupo

Foram encontrados exclusivamente os grupos **1, 2, 3 e 4**, conforme estabelecido pelo dicionário de dados.

A distribuição encontrada foi:

- Grupo 1: 38 equipes
- Grupo 2: 40 equipes
- Grupo 3: 37 equipes
- Grupo 4: 17 equipes

**Conclusão:** foram identificadas inconsistências de padronização nas variáveis `distrito_sanitario` e `tipo_equipe`, já a variável `grupo` está de acordo com o domínio esperado. As categorias inconsistentes serão apenas documentadas nesta etapa, sua padronização será realizada posteriormente durante a Preparação dos Dados.

In [13]:
print(f"Número mínimo de profissionais: {equipes["n_profissionais"].min()}")
print(f"Número máximo de profissionais: {equipes["n_profissionais"].max()}")
print(
    "Todos dentro do domínio 8–80:",
    equipes["n_profissionais"].between(8, 80).all()
)

Número mínimo de profissionais: 8
Número máximo de profissionais: 53
Todos dentro do domínio 8–80: True


In [14]:
equipes["populacao_adscrita"].describe()

count      121.000000
mean      3930.570248
std       2042.458506
min        900.000000
25%       2606.000000
50%       3441.000000
75%       5148.000000
max      10847.000000
Name: populacao_adscrita, dtype: float64

In [15]:
equipes[equipes["populacao_adscrita"] <= 0]

,codigo_cnes,codigo_ine,nome_unidade,nome_equipe,distrito_sanitario,tipo_equipe,grupo,populacao_adscrita,n_profissionais


### Validação das variáveis numéricas

- A variável `n_profissionais` apresentou valores entre **8 e 53 profissionais**, todos dentro do domínio esperado pelo dicionário de dados (8 a 80).
- Para `populacao_adscrita`, considerando os 121 registros preenchidos, foram observados valores entre **900 e 10.847 pessoas**, com mediana de **3.441**. O dicionário não estabelece limites mínimo e máximo para essa variável, portanto esses valores não serão classificados como inconsistentes apenas com base em sua magnitude.
- Os 11 valores ausentes de `populacao_adscrita` identificados anteriormente permanecem documentados para avaliação na etapa de Preparação dos Dados.
- Não foram encontrados valores iguais ou inferiores a zero em `populacao_adscrita`. 

## 2. Tabela resultados_ciclos.csv

## Função auxiliar para perfilamento
Após a análise detalhada da tabela `equipes`, foi criada uma função reutilizável para realizar verificações estruturais básicas nas demais tabelas, reduzindo repetição de código e tornando o processo de análise mais reprodutível.

In [16]:
def perfil_tabela(df, nome):
    print(f"===== {nome.upper()} =====")
    
    print(f"\nDimensão: {df.shape[0]} linhas × {df.shape[1]} colunas")
    
    print("\n------ TIPOS DE DADOS ------")
    print(df.dtypes)
    
    print("\n------ VALORES AUSENTES ------")
    ausentes = pd.DataFrame({
        "quantidade": df.isna().sum(),
        "percentual": (df.isna().mean() * 100).round(2)
    })
    print(ausentes)


## Análise da tabela `resultados_ciclos`

- A tabela `resultados_ciclos` possui granularidade de **uma equipe em um ciclo quadrimestral**.
- A chave esperada é a combinação de `codigo_ine` + `ciclo`.
- Nesta etapa serão avaliadas a estrutura da tabela, completude, unicidade da chave, domínios e regras de negócio associadas aos resultados dos ciclos.

In [17]:
perfil_tabela(resultados, "resultados_ciclos")

===== RESULTADOS_CICLOS =====

Dimensão: 1392 linhas × 13 colunas

------ TIPOS DE DADOS ------
codigo_ine                 int64
ciclo                        str
eixo1_bloco_a            float64
eixo1_bloco_b            float64
eixo1_bloco_c            float64
eixo1_total              float64
taxa_resposta_bloco_a    float64
bloco_a_valido              bool
eixo2_total                  str
eixo3_total              float64
pontuacao_final          float64
zona_classificacao           str
situacao_participacao        str
dtype: object

------ VALORES AUSENTES ------
                       quantidade  percentual
codigo_ine                      0        0.00
ciclo                           0        0.00
eixo1_bloco_a                  58        4.17
eixo1_bloco_b                  58        4.17
eixo1_bloco_c                  58        4.17
eixo1_total                    58        4.17
taxa_resposta_bloco_a           0        0.00
bloco_a_valido                  0        0.00
eixo2_total    

In [18]:
resultados.head(10)

,codigo_ine,ciclo,eixo1_bloco_a,eixo1_bloco_b,eixo1_bloco_c,eixo1_total,taxa_resposta_bloco_a,bloco_a_valido,eixo2_total,eixo3_total,pontuacao_final,zona_classificacao,situacao_participacao
0,6000000000,2023.1,57.3,45.1,40.8,143.2,0.6462,True,122.6,395.8,661.6,Aperfeiçoamento,Participante
1,6000000000,2023.2,43.2,34.8,32.9,110.9,0.5315,True,109.9,357.7,578.5,Aperfeiçoamento,Participante
2,6000000000,2023.3,43.9,32.3,30.7,106.8,0.7081,True,139.5,380.1,626.4,Aperfeiçoamento,Participante
3,6000000000,2024.1,37.4,34.0,29.5,101.0,0.7507,True,133.3,354.3,588.6,Aperfeiçoamento,Participante
4,6000000000,2024.2,41.5,31.2,40.1,112.9,0.5747,True,112.3,327.4,552.6,Aperfeiçoamento,Participante
5,6000000000,2024.3,39.2,31.9,29.3,100.4,0.3710,True,151.8,365.6,617.8,Aperfeiçoamento,Participante
6,6000000000,2025.1,49.7,39.9,42.1,131.7,0.3366,True,133.0,353.8,618.5,Aperfeiçoamento,Participante
7,6000000000,2025.2,NaN,NaN,NaN,NaN,0.3228,True,NaN,NaN,NaN,NaN,Desclassificada
8,6000000000,2025.3,56.8,46.4,38.6,141.9,0.4578,True,131.1,383.9,656.9,Aperfeiçoamento,Participante
9,6000000000,2026.1,50.7,41.5,39.6,131.8,0.6525,True,129.9,417.2,678.9,Aperfeiçoamento,Participante


In [19]:
resultados["situacao_participacao"].value_counts()

situacao_participacao
Participante       1334
Desclassificada      58
Name: count, dtype: int64

In [20]:
colunas_pontuacao = [
    "eixo1_bloco_a",
    "eixo1_bloco_b",
    "eixo1_bloco_c",
    "eixo1_total",
    "eixo3_total",
    "pontuacao_final",
    "zona_classificacao"
]

resultados.groupby("situacao_participacao")[colunas_pontuacao].apply(
    lambda x: x.isna().sum()
)

,eixo1_bloco_a,eixo1_bloco_b,eixo1_bloco_c,eixo1_total,eixo3_total,pontuacao_final,zona_classificacao
situacao_participacao,,,,,,,
Desclassificada,58,58,58,58,58,58,58
Participante,0,0,0,0,0,0,0


### Valores ausentes e situação de participação

- Foram identificados 58 registros classificados como `Desclassificada`.
- Esses mesmos registros apresentam ausência nas variáveis de pontuação e classificação analisadas, enquanto os 1.334 registros classificados como `Participante` possuem `pontuacao_final` preenchida.
- O padrão é coerente com a regra de negócio do Recife Monitora segundo a qual equipes desclassificadas em determinado ciclo não recebem pontuação. Portanto, esses valores ausentes não serão tratados como pontuação zero nem classificados como falha de preenchimento.

In [21]:
resultados.groupby("situacao_participacao")["eixo2_total"].apply(
    lambda x: x.isna().sum()
)

situacao_participacao
Desclassificada     58
Participante       172
Name: eixo2_total, dtype: int64

In [22]:
resultados["eixo2_total"].value_counts(dropna=False)

eixo2_total
NaN      230
131.2      6
122.6      5
125.9      5
115.8      5
        ... 
150.6      1
154.5      1
128,9      1
132,6      1
176.3      1
Name: count, Length: 733, dtype: int64

In [23]:
mascara_virgula = resultados["eixo2_total"].str.contains(",", na=False)
print("Valores com vírgula:", mascara_virgula.sum())

Valores com vírgula: 125


In [24]:
resultados.loc[
    mascara_virgula,
    ["codigo_ine", "ciclo", "eixo2_total"]
].head(10)

,codigo_ine,ciclo,eixo2_total
14,6000000001,2024.1,"102,9"
20,6000000001,2026.1,"109,5"
27,6000000002,2024-3,"198,9"
42,6000000003,2026.1,"123,5"
52,6000000004,2025.3,"106,9"
72,6000000006,2025.1,"148,2"
91,6000000008,2024Q1,"93,5"
98,6000000008,2026.2,"123,4"
120,6000000010,2026.2,"102,0"
136,6000000012,2024.2,"167,2"


In [25]:
participantes_sem_eixo2 = resultados[
    (resultados["situacao_participacao"] == "Participante") &
    (resultados["eixo2_total"].isna())
]

participantes_sem_eixo2[
    [
        "codigo_ine",
        "ciclo",
        "eixo1_total",
        "eixo2_total",
        "eixo3_total",
        "pontuacao_final"
    ]
].head(10)

,codigo_ine,ciclo,eixo1_total,eixo2_total,eixo3_total,pontuacao_final
11,6000000001,2023.1,101.0,NaN,338.6,560.4
12,6000000001,2023-2,123.5,NaN,359.3,595.6
13,6000000001,2023Q3,118.9,NaN,302.0,565.3
22,6000000002,2023.1,128.1,NaN,407.2,693.3
34,6000000003,2023-2,112.2,NaN,405.2,642.7
44,6000000004,2023.1,77.6,NaN,334.8,540.9
45,6000000004,2023.2,126.3,NaN,331.0,594.3
46,6000000004,2023.3,70.5,NaN,330.5,524.0
55,6000000005,2023.1,145.3,NaN,429.3,742.6
56,6000000005,2023.2,140.7,NaN,462.4,742.8


In [26]:
participantes_sem_eixo2.shape

(172, 13)

### Análise da variável `eixo2_total`

- A variável `eixo2_total` apresentou as seguintes inconsistências de completude e formatação: 
- Foram identificados **230 valores ausentes (16,52%)**. Destes **58 pertencem a registros classificados como `Desclassificada`**, sendo coerentes com a ausência de pontuação observada para essa situação de participação.
- Porém foram encontrados **172 registros classificados como `Participante` com `eixo2_total` ausente**, mesmo apresentando `pontuacao_final` preenchida. Essa situação requer investigação adicional antes da definição de qualquer estratégia de tratamento.
- Também foram identificados **125 registros utilizando vírgula como separador decimal**, enquanto os demais valores utilizam ponto. Essa inconsistência de formatação contribui para que a variável seja interpretada como texto (`str`), embora o dicionário de dados estabeleça uma pontuação numérica de 0 a 200.

In [27]:
duplicados_chave = resultados[
    resultados.duplicated(
        subset=["codigo_ine", "ciclo"],
        keep=False
    )
].sort_values(["codigo_ine", "ciclo"])

duplicados_chave



,codigo_ine,ciclo,eixo1_bloco_a,eixo1_bloco_b,eixo1_bloco_c,eixo1_total,taxa_resposta_bloco_a,bloco_a_valido,eixo2_total,eixo3_total,pontuacao_final,zona_classificacao,situacao_participacao
17,6000000001,2025.1,46.4,33.9,32.1,112.4,0.5851,True,120.6,393.5,626.5,Aperfeiçoamento,Participante
1366,6000000001,2025.1,46.4,33.9,32.1,112.4,0.5851,True,120.6,393.5,626.5,Aperfeiçoamento,Participante
517,6000000048,2025.3,31.4,27.0,28.0,86.4,0.5195,True,88.8,302.5,477.7,Crítica,Participante
1375,6000000048,2025.3,31.4,27.0,28.0,86.4,0.5195,True,88.8,302.5,477.7,Crítica,Participante
586,6000000055,2025.3,49.3,38.4,45.2,132.9,0.3866,True,101.9,367.7,602.5,Aperfeiçoamento,Participante
1373,6000000055,2025.3,49.3,38.4,45.2,132.9,0.3866,True,101.9,367.7,602.5,Aperfeiçoamento,Participante
602,6000000057,2023.3,37.2,35.3,30.7,103.2,0.5279,True,NaN,325.0,550.6,Aperfeiçoamento,Participante
1376,6000000057,2023.3,37.2,35.3,30.7,103.2,0.5279,True,122.4,325.0,550.6,Aperfeiçoamento,Participante
638,6000000060,2026.2,0.0,43.2,38.6,81.8,0.2841,False,126.5,408.8,617.1,Aperfeiçoamento,Participante
1364,6000000060,2026.2,0.0,43.2,38.6,81.8,0.2841,False,126.5,408.8,617.1,Aperfeiçoamento,Participante


In [28]:
print(
    "Duplicidades exatas:",
    resultados.duplicated().sum()
)

print(
    "Combinações duplicadas de codigo_ine + ciclo:",
    resultados.duplicated(
        subset=["codigo_ine", "ciclo"]
    ).sum()
)

Duplicidades exatas: 10
Combinações duplicadas de codigo_ine + ciclo: 12


### Validação da chave composta

- A granularidade esperada da tabela `resultados_ciclos` é de **uma equipe por ciclo**, sendo a combinação `codigo_ine + ciclo` a chave esperada.
- Foram identificadas **12 combinações duplicadas de equipe e ciclo**, envolvendo 24 registros. Cada combinação duplicada aparece exatamente duas vezes.
Entre os 12 pares identificados:

- **10 pares são duplicidades exatas**, com os mesmos valores em todas as colunas;
- **2 pares apresentam divergência entre os registros**:
  - `codigo_ine = 6000000057`, ciclo `2023.3`: divergência em `eixo2_total` (`NaN` em um registro e `122.4` no outro);
  - `codigo_ine = 6000000069`, ciclo `2023.2`: divergência em `eixo2_total` (`NaN` em um registro e `170.6` no outro).

- Dessa forma, a chave composta esperada `codigo_ine + ciclo` não apresenta unicidade na base original.

In [29]:
print("Quantidade de ciclos distintos:", resultados["ciclo"].nunique())

print("\nValores encontrados:")
print(resultados["ciclo"].value_counts().sort_index())

Quantidade de ciclos distintos: 43

Valores encontrados:
ciclo
1/2023      4
1/2024      1
1/2025      2
1/2026      3
2/2023      4
2/2024      4
2/2025      5
2/2026      2
2023-1      6
2023-2      2
2023-3      3
2023.1     95
2023.2    104
2023.3    108
2023Q1      2
2023Q2      2
2023Q3      4
2024-1      1
2024-2      3
2024-3      1
2024.1    118
2024.2    118
2024.3    127
2024Q1      4
2024Q2      3
2024Q3      2
2025-1      4
2025-2      1
2025-3      6
2025.1    125
2025.2    124
2025.3    120
2025Q1      4
2025Q2      3
2025Q3      4
2026-1      3
2026-2      4
2026.1    127
2026.2    130
2026Q1      1
2026Q2      2
3/2024      1
3/2025      5
Name: count, dtype: int64


In [30]:
padrao_ciclo = r"^\d{4}\.[1-3]$"

ciclo_valido = resultados["ciclo"].str.match(padrao_ciclo)

print("Registros no formato esperado:", ciclo_valido.sum())
print("Registros fora do formato esperado:", (~ciclo_valido).sum())

Registros no formato esperado: 1296
Registros fora do formato esperado: 96


### Validação do domínio de `ciclo`

- Segundo o dicionário de dados, a variável `ciclo` deve seguir o formato `AAAA.Q`, representando os ciclos quadrimestrais do programa.
- Embora a base represente **11 ciclos conceituais**, entre `2023.1` e `2026.2` foram encontradas **43 representações textuais distintas**.
- Dos 1.392 registros:
    - **1.296 (93,1%)** estão no formato esperado;
    - **96 (6,9%)** estão fora do padrão.
- Foram observadas diferentes representações para um mesmo ciclo, como `2023.1`, `2023-1`, `2023Q1` e `1/2023`.

In [31]:
print("Taxa mínima:", resultados["taxa_resposta_bloco_a"].min())
print("Taxa máxima:", resultados["taxa_resposta_bloco_a"].max())

fora_intervalo = ~resultados["taxa_resposta_bloco_a"].between(0, 1)

print(
    "Registros fora do intervalo 0–1:",
    fora_intervalo.sum()
)

Taxa mínima: -0.1
Taxa máxima: 2.4
Registros fora do intervalo 0–1: 7


In [32]:
resultados.loc[
    fora_intervalo,
    [
        "codigo_ine",
        "ciclo",
        "taxa_resposta_bloco_a",
        "bloco_a_valido",
        "eixo1_bloco_a"
    ]
]

,codigo_ine,ciclo,taxa_resposta_bloco_a,bloco_a_valido,eixo1_bloco_a
261,6000000025,2024.3,2.40,True,50.3
298,6000000028,2026.1,-0.10,True,56.3
694,6000000066,2023.2,1.35,True,35.7
1045,6000000101,2025.1,2.40,True,66.8
1127,6000000109,2023.2,-0.10,True,58.2
1278,6000000123,2025.3,-0.10,True,60.8
1311,6000000127,2023.2,-0.10,True,43.8


In [33]:
bloco_a_esperado = resultados["taxa_resposta_bloco_a"] >= 0.30

inconsistencia_bloco_a = (
    resultados["bloco_a_valido"] != bloco_a_esperado
)

print(
    "Registros inconsistentes com a regra dos 30%:",
    inconsistencia_bloco_a.sum()
)

Registros inconsistentes com a regra dos 30%: 4


In [34]:
resultados.loc[
    resultados["bloco_a_valido"] == False,
    ["taxa_resposta_bloco_a", "bloco_a_valido", "eixo1_bloco_a"]
].head(10)

,taxa_resposta_bloco_a,bloco_a_valido,eixo1_bloco_a
14,0.2942,False,0.0
29,0.2242,False,0.0
37,0.2903,False,0.0
42,0.1696,False,0.0
44,0.1755,False,0.0
54,0.1543,False,0.0
88,0.2045,False,0.0
104,0.2155,False,0.0
107,0.2071,False,0.0
108,0.2982,False,0.0


In [35]:
resultados.loc[
    resultados["bloco_a_valido"] == False,
    "eixo1_bloco_a"
].value_counts(dropna=False)

eixo1_bloco_a
0.0    152
NaN      6
Name: count, dtype: int64

### Validação da taxa de resposta e do Bloco A

- A variável `taxa_resposta_bloco_a` deve assumir valores entre 0 e 1. Mas foram identificados **7 registros fora desse domínio** com valores variando entre **-0,10 e 2,40**.
- Também foi verificada a regra de negócio segundo a qual o `bloco_a_valido` deve ser `True` quando a taxa de resposta for maior ou igual a 30% (`0,30`).
- Foram encontrados **4 registros inconsistentes com essa regra**, todos apresentando taxa de `-0,10` e `bloco_a_valido = True`, quando o esperado seria `False`.
- Adicionalmente, foram analisados os registros em que `bloco_a_valido = False` e foram encontrados **158 registros** dos quais **152 apresentam `eixo1_bloco_a = 0`** e **6 apresentam valor ausente (`NaN`)**.
- Essa diferença indica uma inconsistência na forma de representar a pontuação de um Bloco A inválido. Como a regra de negócio informa que a pontuação não deve ser computada quando a taxa de resposta é inferior a 30%, a interpretação entre valor zero e ausência de pontuação deverá ser confirmada antes da definição do tratamento.

In [36]:
colunas_limites = {
    "eixo1_total": (0, 200),
    "eixo3_total": (0, 600),
    "pontuacao_final": (0, 1000)
}

for coluna, (minimo, maximo) in colunas_limites.items():
    preenchidos = resultados[coluna].dropna()
    
    fora = ~preenchidos.between(minimo, maximo)
    
    print(
        coluna,
        "| mínimo:", preenchidos.min(),
        "| máximo:", preenchidos.max(),
        "| fora do limite:", fora.sum()
    )

eixo1_total | mínimo: 32.2 | máximo: 199.3 | fora do limite: 0
eixo3_total | mínimo: 183.8 | máximo: 559.9 | fora do limite: 0
pontuacao_final | mínimo: -35.0 | máximo: 9999.0 | fora do limite: 9


In [37]:
pontuacao_fora_limite = resultados[
    resultados["pontuacao_final"].notna() &
    ~resultados["pontuacao_final"].between(0, 1000)
]

pontuacao_fora_limite[
    [
        "codigo_ine",
        "ciclo",
        "eixo1_total",
        "eixo2_total",
        "eixo3_total",
        "pontuacao_final",
        "zona_classificacao",
        "situacao_participacao"
    ]
]

,codigo_ine,ciclo,eixo1_total,eixo2_total,eixo3_total,pontuacao_final,zona_classificacao,situacao_participacao
29,6000000002,2/2025,89.5,191.5,465.4,1247.0,Qualidade,Participante
96,6000000008,2025.3,86.3,110.2,317.6,1580.0,Crítica,Participante
247,6000000024,2023.3,86.0,124.5,349.5,1580.0,Aperfeiçoamento,Participante
291,6000000028,2023.3,97.9,NaN,323.5,-35.0,Aperfeiçoamento,Participante
502,6000000047,2024Q2,128.9,114.3,389.2,9999.0,Aperfeiçoamento,Participante
1010,6000000098,2023.2,77.7,63.3,219.2,-35.0,Insuficiente,Participante
1139,6000000110,2023.3,123.5,NaN,367.5,-35.0,Aperfeiçoamento,Participante
1213,6000000117,2025.1,119.9,125.5,365.4,1247.0,Aperfeiçoamento,Participante
1217,6000000117,2026-2,116.2,117.4,413.8,9999.0,Aperfeiçoamento,Participante


In [38]:
def classificar_zona(pontuacao):
    if pd.isna(pontuacao):
        return np.nan
    elif pontuacao >= 850:
        return "Excelência"
    elif pontuacao >= 700:
        return "Qualidade"
    elif pontuacao >= 550:
        return "Aperfeiçoamento"
    elif pontuacao >= 400:
        return "Crítica"
    else:
        return "Insuficiente"

zona_esperada = resultados["pontuacao_final"].apply(classificar_zona)

inconsistencia_zona = (
    resultados["pontuacao_final"].notna() &
    (resultados["zona_classificacao"] != zona_esperada)
)

print(
    "Registros com classificação inconsistente:",
    inconsistencia_zona.sum()
)

Registros com classificação inconsistente: 8


In [39]:
resultados.loc[
    inconsistencia_zona,
    [
        "codigo_ine",
        "ciclo",
        "pontuacao_final",
        "zona_classificacao"
    ]
].assign(
    zona_esperada=zona_esperada[inconsistencia_zona]
)


,codigo_ine,ciclo,pontuacao_final,zona_classificacao,zona_esperada
29,6000000002,2/2025,1247.0,Qualidade,Excelência
96,6000000008,2025.3,1580.0,Crítica,Excelência
247,6000000024,2023.3,1580.0,Aperfeiçoamento,Excelência
291,6000000028,2023.3,-35.0,Aperfeiçoamento,Insuficiente
502,6000000047,2024Q2,9999.0,Aperfeiçoamento,Excelência
1139,6000000110,2023.3,-35.0,Aperfeiçoamento,Insuficiente
1213,6000000117,2025.1,1247.0,Aperfeiçoamento,Excelência
1217,6000000117,2026-2,9999.0,Aperfeiçoamento,Excelência


### Validação dos limites de pontuação e da zona de classificação

Foram avaliados os limites definidos no dicionário de dados para as pontuações.

- Entre os valores preenchidos:
    - `eixo1_total` apresentou valores entre **32,2 e 199,3**, sem registros fora do intervalo esperado de 0 a 200;
    - `eixo3_total` apresentou valores entre **183,8 e 559,9**, sem registros fora do intervalo esperado de 0 a 600;
    - `pontuacao_final` apresentou **9 registros fora do intervalo esperado de 0 a 1.000**, incluindo valores negativos e valores superiores ao limite máximo, como `-35`, `1247`, `1580` e `9999`.
- Também foi recalculada a zona de classificação a partir das faixas definidas no dicionário e comparada com a variável `zona_classificacao` existente na base.Foram identificados **8 registros em que a classificação armazenada não corresponde à classificação esperada a partir da `pontuacao_final`**.

As inconsistências foram apenas identificadas e documentadas nesta etapa, sem alteração dos valores originais.

In [40]:
eixo2_numerico = pd.to_numeric(
    resultados["eixo2_total"].str.replace(",", ".", regex=False),
    errors="coerce"
)

soma_eixos = (
    resultados["eixo1_total"] +
    eixo2_numerico +
    resultados["eixo3_total"]
)

diferenca_pontuacao = (
    resultados["pontuacao_final"] - soma_eixos
).abs()

inconsistencia_soma = (
    soma_eixos.notna() &
    resultados["pontuacao_final"].notna() &
    (diferenca_pontuacao > 0.01)
)

print(
    "Registros em que a pontuação final não corresponde à soma dos eixos:",
    inconsistencia_soma.sum()
)


Registros em que a pontuação final não corresponde à soma dos eixos: 7


In [41]:
comparacao_soma = resultados.loc[
    inconsistencia_soma,
    [
        "codigo_ine",
        "ciclo",
        "eixo1_total",
        "eixo2_total",
        "eixo3_total",
        "pontuacao_final"
    ]
].copy()

comparacao_soma["soma_eixos"] = soma_eixos[inconsistencia_soma]
comparacao_soma["diferenca"] = diferenca_pontuacao[inconsistencia_soma]

comparacao_soma

,codigo_ine,ciclo,eixo1_total,eixo2_total,eixo3_total,pontuacao_final,soma_eixos,diferenca
29,6000000002,2/2025,89.5,191.5,465.4,1247.0,746.4,500.6
96,6000000008,2025.3,86.3,110.2,317.6,1580.0,514.1,1065.9
247,6000000024,2023.3,86.0,124.5,349.5,1580.0,560.0,1020.0
502,6000000047,2024Q2,128.9,114.3,389.2,9999.0,632.4,9366.6
1010,6000000098,2023.2,77.7,63.3,219.2,-35.0,360.2,395.2
1213,6000000117,2025.1,119.9,125.5,365.4,1247.0,610.8,636.2
1217,6000000117,2026-2,116.2,117.4,413.8,9999.0,647.4,9351.6


### Consistência entre os eixos e a pontuação final

- Para os registros com valores disponíveis nos três eixos, foi calculada a soma de `eixo1_total`, `eixo2_total` e `eixo3_total` e comparada com a `pontuacao_final`.
- Foram identificados **7 registros em que a pontuação final não corresponde à soma dos três eixos**.
- As diferenças observadas são substanciais, não sendo compatíveis apenas com efeitos de arredondamento, alguns desses registros também estão entre os casos anteriormente identificados com `pontuacao_final` fora do intervalo permitido de 0 a 1.000.
- Para possibilitar essa validação, foi criada apenas uma representação numérica auxiliar de `eixo2_total`, substituindo temporariamente a vírgula pelo ponto como separador decimal, sem alteração da coluna original.


## 3. Tabela eixo3_indicadores.csv

In [42]:
perfil_tabela(eixo3, "eixo3_indicadores")

===== EIXO3_INDICADORES =====

Dimensão: 13294 linhas × 7 colunas

------ TIPOS DE DADOS ------
codigo_ine                int64
ciclo                   float64
codigo_indicador            str
indicador                   str
peso_maximo               int64
percentual_alcancado    float64
pontos_obtidos          float64
dtype: object

------ VALORES AUSENTES ------
                      quantidade  percentual
codigo_ine                     0         0.0
ciclo                          0         0.0
codigo_indicador               0         0.0
indicador                      0         0.0
peso_maximo                    0         0.0
percentual_alcancado           0         0.0
pontos_obtidos                 0         0.0


In [43]:
eixo3.head()

,codigo_ine,ciclo,codigo_indicador,indicador,peso_maximo,percentual_alcancado,pontos_obtidos
0,6000000000,2023.1,IND01,Proporção de gestantes com 6 ou mais consultas...,75,0.6223,46.7
1,6000000000,2023.1,IND02,Proporção de gestantes com exames de sífilis e...,70,0.7733,54.1
2,6000000000,2023.1,IND03,Proporção de gestantes com atendimento odontol...,55,0.8604,47.3
3,6000000000,2023.1,IND04,Cobertura de exame citopatológico do colo do ú...,70,0.6073,42.5
4,6000000000,2023.1,IND05,Cobertura vacinal de poliomielite e pentavalen...,75,0.6256,46.9


In [44]:
print(
    "Duplicidades da chave composta:",
    eixo3.duplicated(
        subset=["codigo_ine", "ciclo", "codigo_indicador"]
    ).sum()
)

print(
    "Duplicidades exatas:",
    eixo3.duplicated().sum()
)

Duplicidades da chave composta: 0
Duplicidades exatas: 0


In [45]:
print("Ciclos distintos:", eixo3["ciclo"].nunique())
print("Indicadores distintos:", eixo3["codigo_indicador"].nunique())

print("\nCiclos encontrados:")
print(sorted(eixo3["ciclo"].unique()))

print("\nIndicadores encontrados:")
print(eixo3["codigo_indicador"].value_counts().sort_index())

Ciclos distintos: 11
Indicadores distintos: 10

Ciclos encontrados:
[np.float64(2023.1), np.float64(2023.2), np.float64(2023.3), np.float64(2024.1), np.float64(2024.2), np.float64(2024.3), np.float64(2025.1), np.float64(2025.2), np.float64(2025.3), np.float64(2026.1), np.float64(2026.2)]

Indicadores encontrados:
codigo_indicador
IND01    1363
IND02    1363
IND03    1195
IND04    1363
IND05    1363
IND06    1363
IND07    1363
IND08    1195
IND09    1363
IND10    1363
Name: count, dtype: int64


### Validação da granularidade e dos domínios

- A tabela `eixo3_indicadores` possui 13.294 registros e não apresenta valores ausentes.
- A chave composta esperada (`codigo_ine + ciclo + codigo_indicador`) não apresentou duplicidades, confirmando a granularidade de um indicador por equipe e ciclo.
- Foram encontrados **11 ciclos distintos**, correspondentes ao período de `2023.1` a `2026.2` e **10 códigos de indicadores** de `IND01` a `IND10`, conforme previsto no dicionário de dados.
- A quantidade de registros varia entre os indicadores. Essa diferença não foi considerada uma inconsistência neste momento, pois a aplicabilidade dos indicadores do Eixo 3 depende do grupo da equipe.
- A variável `ciclo` foi interpretada pelo Pandas como `float64`. Embora os valores observados correspondam aos 11 ciclos esperados, conceitualmente essa variável representa um identificador temporal e deverá ser tratada adequadamente na etapa de Preparação dos Dados.

In [46]:
print(
    "Percentual mínimo:",
    eixo3["percentual_alcancado"].min()
)

print(
    "Percentual máximo:",
    eixo3["percentual_alcancado"].max()
)

print(
    "Percentuais fora de 0–1:",
    (~eixo3["percentual_alcancado"].between(0, 1)).sum()
)

print(
    "Pesos iguais ou menores que zero:",
    (eixo3["peso_maximo"] <= 0).sum()
)

print(
    "Pontos obtidos menores que zero:",
    (eixo3["pontos_obtidos"] < 0).sum()
)

Percentual mínimo: 0.05
Percentual máximo: 100.0
Percentuais fora de 0–1: 380
Pesos iguais ou menores que zero: 0
Pontos obtidos menores que zero: 0


In [47]:
percentuais_fora = eixo3[
    ~eixo3["percentual_alcancado"].between(0, 1)
]

percentuais_fora[
    [
        "codigo_ine",
        "ciclo",
        "codigo_indicador",
        "peso_maximo",
        "percentual_alcancado",
        "pontos_obtidos"
    ]
].head(15)

,codigo_ine,ciclo,codigo_indicador,peso_maximo,percentual_alcancado,pontos_obtidos
46,6000000000,2024.2,IND07,60,55.01,33.0
65,6000000000,2025.1,IND06,65,74.65,48.5
75,6000000000,2025.2,IND06,65,48.69,31.6
103,6000000000,2026.2,IND04,70,62.16,43.5
114,6000000001,2023.1,IND05,80,51.52,41.2
116,6000000001,2023.1,IND07,65,60.45,39.3
142,6000000001,2024.1,IND03,45,51.38,23.1
221,6000000002,2023.1,IND02,65,72.80,47.3
231,6000000002,2023.2,IND02,65,86.38,56.1
259,6000000002,2024.1,IND10,40,69.31,27.7


In [48]:
pontos_esperados_fora = (
    percentuais_fora["peso_maximo"] *
    (percentuais_fora["percentual_alcancado"] / 100)
)

diferenca = (
    percentuais_fora["pontos_obtidos"] - pontos_esperados_fora
).abs()

print(
    "Compatíveis com escala 0–100:",
    (diferenca <= 0.15).sum()
)

print(
    "Diferença superior à tolerância:",
    (diferenca > 0.15).sum()
)

Compatíveis com escala 0–100: 380
Diferença superior à tolerância: 0


### Validação de `percentual_alcancado`

- Segundo o dicionário de dados, `percentual_alcancado` deve ser representado em escala de 0 a 1.
- Foram identificados **380 registros acima de 1**, com valores chegando a 100. A análise conjunta com `peso_maximo` e `pontos_obtidos` indicou que esses registros utilizam na prática uma representação percentual em escala de 0 a 100.Por exemplo, um registro com `peso_maximo = 60` e `percentual_alcancado = 55,01` apresenta `pontos_obtidos = 33,0`, valor compatível com o cálculo `60 × 55,01%`.
- A comparação sistemática confirmou que os registros são compatíveis com essa interpretação, considerando pequenas diferenças decorrentes de arredondamento. Dessa forma, foi identificada uma **mistura de escalas (0–1 e 0–100)** na variável `percentual_alcancado`. 

In [49]:
percentual_padronizado = eixo3["percentual_alcancado"].copy()

percentual_padronizado = percentual_padronizado.where(
    percentual_padronizado <= 1,
    percentual_padronizado / 100
)

pontos_esperados = (
    eixo3["peso_maximo"] * percentual_padronizado
)

diferenca_pontos = (
    eixo3["pontos_obtidos"] - pontos_esperados
).abs()

print(
    "Registros compatíveis com o cálculo:",
    (diferenca_pontos <= 0.15).sum()
)

print(
    "Registros com diferença superior à tolerância:",
    (diferenca_pontos > 0.15).sum()
)

Registros compatíveis com o cálculo: 13294
Registros com diferença superior à tolerância: 0


- Após a padronização temporária da escala apenas para fins de validação, foi verificada a relação entre `peso_maximo`, `percentual_alcancado` e `pontos_obtidos`.
- Os **13.294 registros (100%)** apresentaram pontuação compatível com o cálculo esperado, considerando tolerância para arredondamento.
- Esse resultado reforça que a principal inconsistência identificada em `percentual_alcancado` está relacionada à **representação em escalas diferentes (0–1 e 0–100)**, e não ao cálculo de `pontos_obtidos`.

In [50]:
ines_eixo3 = set(eixo3["codigo_ine"])
ines_equipes = set(equipes["codigo_ine"])

ines_sem_equipe = ines_eixo3 - ines_equipes

print("INEs distintos no Eixo 3:", len(ines_eixo3))
print("INEs sem correspondência em equipes:", len(ines_sem_equipe))

print("INEs sem correspondência:", ines_sem_equipe)

INEs distintos no Eixo 3: 132
INEs sem correspondência em equipes: 0
INEs sem correspondência: set()


In [51]:
eixo3_com_grupo = eixo3.merge(
    equipes[["codigo_ine", "grupo"]],
    on="codigo_ine",
    how="left"
)

eixo3_com_grupo.head()

,codigo_ine,ciclo,codigo_indicador,indicador,peso_maximo,percentual_alcancado,pontos_obtidos,grupo
0,6000000000,2023.1,IND01,Proporção de gestantes com 6 ou mais consultas...,75,0.6223,46.7,2
1,6000000000,2023.1,IND02,Proporção de gestantes com exames de sífilis e...,70,0.7733,54.1,2
2,6000000000,2023.1,IND03,Proporção de gestantes com atendimento odontol...,55,0.8604,47.3,2
3,6000000000,2023.1,IND04,Cobertura de exame citopatológico do colo do ú...,70,0.6073,42.5,2
4,6000000000,2023.1,IND05,Cobertura vacinal de poliomielite e pentavalen...,75,0.6256,46.9,2


In [52]:
pesos_por_grupo = (
    eixo3_com_grupo
    .groupby(["grupo", "codigo_indicador"])["peso_maximo"]
    .unique()
)

pesos_por_grupo

grupo  codigo_indicador
1      IND01               [70]
       IND02               [65]
       IND03               [60]
       IND04               [65]
       IND05               [70]
       IND06               [65]
       IND07               [60]
       IND08               [55]
       IND09               [50]
       IND10               [40]
2      IND01               [75]
       IND02               [70]
       IND03               [55]
       IND04               [70]
       IND05               [75]
       IND06               [65]
       IND07               [60]
       IND08               [50]
       IND09               [45]
       IND10               [35]
3      IND01               [80]
       IND02               [75]
       IND03               [45]
       IND04               [75]
       IND05               [80]
       IND06               [70]
       IND07               [65]
       IND08               [40]
       IND09               [40]
       IND10               [30]
4      IND01    

In [53]:
pesos_grupo_indicador = (
    eixo3_com_grupo
    .groupby(["grupo", "codigo_indicador"])["peso_maximo"]
    .first()
    .reset_index()
)

soma_pesos_por_grupo = (
    pesos_grupo_indicador
    .groupby("grupo")["peso_maximo"]
    .sum()
)

soma_pesos_por_grupo

grupo
1    600
2    600
3    600
4    600
Name: peso_maximo, dtype: int64

In [54]:
soma_pesos_equipe_ciclo = (
    eixo3_com_grupo
    .groupby(["codigo_ine", "ciclo"])["peso_maximo"]
    .sum()
)

print("Quantidade de equipe × ciclo:", len(soma_pesos_equipe_ciclo))

print(
    "Equipe × ciclo com soma diferente de 600:",
    (soma_pesos_equipe_ciclo != 600).sum()
)

soma_pesos_equipe_ciclo.value_counts().sort_index()

Quantidade de equipe × ciclo: 1363
Equipe × ciclo com soma diferente de 600: 0


peso_maximo
600    1363
Name: count, dtype: int64

### Validação dos indicadores e pesos por grupo
- A aplicabilidade e os pesos dos indicadores do Eixo 3 foram avaliados em conjunto com o grupo de cada equipe, obtido por meio do cruzamento com a tabela `equipes`.
- Foi verificado que os pesos variam conforme o grupo da equipe. Os grupos 1, 2 e 3 apresentam os dez indicadores (`IND01` a `IND10`), enquanto o grupo 4 não possui os indicadores `IND03` e `IND08`, comportamento compatível com a regra de aplicabilidade diferenciada.
- A soma dos pesos máximos dos indicadores aplicáveis corresponde a **600 pontos em todos os quatro grupos**.
- Também foi realizada a validação por equipe e ciclo. Foram identificadas **1.363 combinações de equipe × ciclo**, e todas apresentaram soma dos pesos igual a **600 pontos**, sem inconsistências nessa regra de negócio.

## Tabela nps_atributos.csv

In [55]:
perfil_tabela(nps, "nps_atributos")

===== NPS_ATRIBUTOS =====

Dimensão: 6815 linhas × 6 colunas

------ TIPOS DE DADOS ------
codigo_ine           int64
ciclo              float64
codigo_atributo        str
atributo               str
nps                float64
n_respondentes       int64
dtype: object

------ VALORES AUSENTES ------
                 quantidade  percentual
codigo_ine                0        0.00
ciclo                     0        0.00
codigo_atributo           0        0.00
atributo                  0        0.00
nps                     260        3.82
n_respondentes            0        0.00


In [56]:
nps.head()

,codigo_ine,ciclo,codigo_atributo,atributo,nps,n_respondentes
0,6000000000,2023.1,ATR1,Acolhimento da equipe,44.6,21
1,6000000000,2023.1,ATR2,Tempo de espera para atendimento,15.4,28
2,6000000000,2023.1,ATR3,Resolutividade do atendimento,27.9,17
3,6000000000,2023.1,ATR4,Estrutura e limpeza da unidade,23.7,11
4,6000000000,2023.1,ATR5,Vínculo e confiança com a equipe,1.4,52


In [57]:
print(
    "Duplicidades da chave composta:",
    nps.duplicated(
        subset=["codigo_ine", "ciclo", "codigo_atributo"]
    ).sum()
)

print(
    "Duplicidades exatas:",
    nps.duplicated().sum()
)

print("\nCiclos distintos:", nps["ciclo"].nunique())
print("Atributos distintos:", nps["codigo_atributo"].nunique())

print("\nAtributos encontrados:")
print(nps["codigo_atributo"].value_counts().sort_index())

Duplicidades da chave composta: 0
Duplicidades exatas: 0

Ciclos distintos: 11
Atributos distintos: 5

Atributos encontrados:
codigo_atributo
ATR1    1363
ATR2    1363
ATR3    1363
ATR4    1363
ATR5    1363
Name: count, dtype: int64


In [58]:
nps_preenchido = nps["nps"].dropna()

print("NPS mínimo:", nps_preenchido.min())
print("NPS máximo:", nps_preenchido.max())

print(
    "NPS fora do intervalo -100 a 100:",
    (~nps_preenchido.between(-100, 100)).sum()
)

print(
    "Respondentes negativos:",
    (nps["n_respondentes"] < 0).sum()
)

NPS mínimo: -70.0
NPS máximo: 100.0
NPS fora do intervalo -100 a 100: 0
Respondentes negativos: 0


In [59]:
nps_sem_valor = nps[nps["nps"].isna()]

print("Registros com NPS ausente:", len(nps_sem_valor))

print("\nResumo de respondentes nesses registros:")
print(nps_sem_valor["n_respondentes"].describe())

Registros com NPS ausente: 260

Resumo de respondentes nesses registros:
count    260.000000
mean      42.596154
std       24.464030
min        3.000000
25%       25.000000
50%       40.000000
75%       57.000000
max      127.000000
Name: n_respondentes, dtype: float64


In [60]:
nps_sem_valor[
    [
        "codigo_ine",
        "ciclo",
        "codigo_atributo",
        "nps",
        "n_respondentes"
    ]
].head(15)

,codigo_ine,ciclo,codigo_atributo,nps,n_respondentes
62,6000000001,2023.2,ATR3,NaN,44
93,6000000001,2025.2,ATR4,NaN,62
231,6000000004,2023.3,ATR2,NaN,60
239,6000000004,2024.1,ATR5,NaN,35
269,6000000004,2026.1,ATR5,NaN,49
297,6000000005,2024.2,ATR3,NaN,121
331,6000000006,2023.1,ATR2,NaN,55
347,6000000006,2024.1,ATR3,NaN,57
360,6000000006,2025.1,ATR1,NaN,61
380,6000000006,2026.2,ATR1,NaN,60


In [61]:
ines_nps = set(nps["codigo_ine"])
ines_equipes = set(equipes["codigo_ine"])

ines_nps_sem_equipe = ines_nps - ines_equipes

print("INEs distintos no NPS:", len(ines_nps))
print("INEs sem correspondência em equipes:", len(ines_nps_sem_equipe))
print("INEs sem correspondência:", ines_nps_sem_equipe)

INEs distintos no NPS: 132
INEs sem correspondência em equipes: 0
INEs sem correspondência: set()


In [62]:
atributos_por_equipe_ciclo = (
    nps.groupby(["codigo_ine", "ciclo"])["codigo_atributo"]
    .nunique()
)

print(
    "Quantidade de equipe × ciclo:",
    len(atributos_por_equipe_ciclo)
)

print(
    "Equipe × ciclo com quantidade diferente de 5 atributos:",
    (atributos_por_equipe_ciclo != 5).sum()
)

Quantidade de equipe × ciclo: 1363
Equipe × ciclo com quantidade diferente de 5 atributos: 0


### Validação da tabela `nps_atributos`

- A tabela `nps_atributos` possui 6.815 registros e apresenta granularidade de **um atributo de satisfação por equipe e ciclo**.
- A chave composta esperada (`codigo_ine + ciclo + codigo_atributo`) não apresentou duplicidades. Foram encontrados os **11 ciclos esperados** e os **5 atributos (`ATR1` a `ATR5`)**, cada um presente em 1.363 combinações de equipe e ciclo.
- Todas as **1.363 combinações equipe × ciclo possuem os cinco atributos esperados**, não sendo identificadas estruturas incompletas.
- Os valores preenchidos de `nps` estão dentro do intervalo esperado de **-100 a 100**, e não foram encontrados valores negativos em `n_respondentes`.
- Foram identificados **260 registros (3,82%) com `nps` ausente**. Esses casos possuem respondentes registrados, variando entre **3 e 127**, indicando que a ausência do NPS não pode ser explicada simplesmente pela inexistência de respostas. A causa deverá ser investigada antes da definição do tratamento.
- Também foi verificada a integridade referencial com a tabela `equipes`: os **132 INEs distintos** presentes em `nps_atributos` possuem correspondência no cadastro de equipes, sem registros órfãos.

## Tabela planos_acao.csv

In [63]:
perfil_tabela(planos, "planos_acao")

===== PLANOS_ACAO =====

Dimensão: 420 linhas × 9 colunas

------ TIPOS DE DADOS ------
id_plano                 str
codigo_ine             int64
ciclo                float64
texto_plano              str
dim_governanca         int64
dim_smart              int64
dim_viabilidade        int64
nivel_maturidade       int64
rotulo_maturidade        str
dtype: object

------ VALORES AUSENTES ------
                   quantidade  percentual
id_plano                    0        0.00
codigo_ine                  0        0.00
ciclo                       0        0.00
texto_plano                10        2.38
dim_governanca              0        0.00
dim_smart                   0        0.00
dim_viabilidade             0        0.00
nivel_maturidade            0        0.00
rotulo_maturidade           0        0.00


In [64]:
planos.head()

,id_plano,codigo_ine,ciclo,texto_plano,dim_governanca,dim_smart,dim_viabilidade,nivel_maturidade,rotulo_maturidade
0,PA00001,6000000058,2024.2,Diante da identificação de alto tempo de esper...,4,5,3,3,Aprovado
1,PA00002,6000000015,2025.2,Melhorar baixa cobertura de primeira consulta ...,2,2,2,1,Reformulação
2,PA00003,6000000045,2024.2,A equipe identificou baixa cobertura de primei...,3,2,4,2,Ajustes
3,PA00004,6000000053,2025.2,A equipe identificou acompanhamento irregular ...,2,2,4,2,Ajustes
4,PA00005,6000000063,2025.2,Diante da identificação de alto tempo de esper...,3,4,3,3,Aprovado


In [65]:
print(
    "IDs de plano duplicados:",
    planos["id_plano"].duplicated().sum()
)

print(
    "Duplicidades exatas:",
    planos.duplicated().sum()
)

print("\nRótulos encontrados:")
print(planos["rotulo_maturidade"].value_counts())

print("\nNíveis de maturidade encontrados:")
print(planos["nivel_maturidade"].value_counts().sort_index())

IDs de plano duplicados: 0
Duplicidades exatas: 0

Rótulos encontrados:
rotulo_maturidade
Ajustes         170
Aprovado        128
Reformulação    122
Name: count, dtype: int64

Níveis de maturidade encontrados:
nivel_maturidade
1    122
2    170
3    128
Name: count, dtype: int64


In [66]:
dimensoes = [
    "dim_governanca",
    "dim_smart",
    "dim_viabilidade"
]

for coluna in dimensoes:
    print(
        coluna,
        "| mínimo:", planos[coluna].min(),
        "| máximo:", planos[coluna].max(),
        "| fora de 1–5:", (~planos[coluna].between(1, 5)).sum()
    )

print(
    "\nNível de maturidade fora de 1–3:",
    (~planos["nivel_maturidade"].between(1, 3)).sum()
)

dim_governanca | mínimo: 1 | máximo: 4 | fora de 1–5: 0
dim_smart | mínimo: 1 | máximo: 5 | fora de 1–5: 0
dim_viabilidade | mínimo: 1 | máximo: 4 | fora de 1–5: 0

Nível de maturidade fora de 1–3: 0


In [67]:
rotulo_esperado = planos["nivel_maturidade"].map({
    1: "Reformulação",
    2: "Ajustes",
    3: "Aprovado"
})

inconsistencia_rotulo = (
    planos["rotulo_maturidade"] != rotulo_esperado
)

print(
    "Inconsistências entre nível e rótulo:",
    inconsistencia_rotulo.sum()
)

Inconsistências entre nível e rótulo: 0


In [68]:
ines_planos = set(planos["codigo_ine"])
ines_equipes = set(equipes["codigo_ine"])

ines_planos_sem_equipe = ines_planos - ines_equipes

print("INEs distintos nos planos:", len(ines_planos))
print(
    "INEs sem correspondência em equipes:",
    len(ines_planos_sem_equipe)
)
print(
    "INEs sem correspondência:",
    ines_planos_sem_equipe
)

INEs distintos nos planos: 128
INEs sem correspondência em equipes: 0
INEs sem correspondência: set()


In [69]:
planos.loc[
    planos["texto_plano"].isna(),
    [
        "id_plano",
        "codigo_ine",
        "ciclo",
        "nivel_maturidade",
        "rotulo_maturidade"
    ]
]

,id_plano,codigo_ine,ciclo,nivel_maturidade,rotulo_maturidade
9,PA00010,6000000097,2025.1,3,Aprovado
64,PA00065,6000000009,2026.1,3,Aprovado
65,PA00066,6000000042,2023.3,1,Reformulação
149,PA00150,6000000087,2024.2,3,Aprovado
179,PA00180,6000000090,2023.1,1,Reformulação
234,PA00235,6000000029,2024.2,2,Ajustes
293,PA00294,6000000131,2026.1,3,Aprovado
331,PA00332,6000000077,2026.2,1,Reformulação
393,PA00394,6000000117,2025.1,2,Ajustes
397,PA00398,6000000010,2025.2,1,Reformulação


In [70]:
chaves_eixo3 = set(
    zip(
        eixo3["codigo_ine"],
        eixo3["ciclo"]
    )
)

chaves_planos = set(
    zip(
        planos["codigo_ine"],
        planos["ciclo"]
    )
)

chaves_planos_sem_correspondencia = (
    chaves_planos - chaves_eixo3
)

print(
    "Combinações equipe × ciclo distintas nos planos:",
    len(chaves_planos)
)

print(
    "Combinações sem correspondência no Eixo 3:",
    len(chaves_planos_sem_correspondencia)
)

print(
    "Combinações sem correspondência:",
    chaves_planos_sem_correspondencia
)

Combinações equipe × ciclo distintas nos planos: 420
Combinações sem correspondência no Eixo 3: 0
Combinações sem correspondência: set()


### Validação da tabela `planos_acao`

- A tabela `planos_acao` possui **420 registros**, com granularidade de **um plano de ação por linha**.
- A chave primária `id_plano` não apresentou duplicidades, assim como não foram identificadas duplicidades exatas na tabela.
- As variáveis `dim_governanca`, `dim_smart` e `dim_viabilidade` apresentaram valores dentro do intervalo esperado de **1 a 5**, enquanto `nivel_maturidade` apresentou somente valores entre **1 e 3**.
- Também foi validada a correspondência entre nível e rótulo de maturidade:
    - nível 1 → `Reformulação`;
    - nível 2 → `Ajustes`;
    - nível 3 → `Aprovado`.

Não foram identificadas inconsistências nessa relação.
- Foram encontrados **10 registros (2,38%) com `texto_plano` ausente**. Como essa variável representa o conteúdo do plano de ação, os casos foram registrados como problema de completude e deverão ser avaliados posteriormente na etapa de Preparação dos Dados.
- A integridade referencial também foi validada. Os **128 INEs distintos** presentes nos planos possuem correspondência na tabela `equipes`. Além disso, as **420 combinações distintas de equipe × ciclo** possuem correspondência no Eixo 3, não sendo identificados registros órfãos.

# Síntese do Entendimento dos Dados

- Nesta etapa foi realizado o perfilamento inicial das cinco tabelas da base sintética do Recife Monitora, com o objetivo de compreender sua estrutura, granularidade, completude, domínios, chaves e principais regras de negócio antes da realização de qualquer tratamento.
- A análise permitiu identificar tanto aspectos consistentes da base quanto problemas de qualidade que deverão ser tratados na etapa de Preparação dos Dados.

> Os dados utilizados nesta análise são sintéticos e não representam resultados reais das equipes, unidades ou profissionais da Secretaria de Saúde do Recife.

## 1. Equipes

- A tabela `equipes` possui **132 registros**, representando uma equipe de saúde por linha.
- A chave `codigo_ine` apresentou **132 valores distintos e nenhuma duplicidade**, confirmando a granularidade esperada.
- Foram identificados **11 valores ausentes (8,33%) em `populacao_adscrita`**. As demais variáveis não apresentaram valores nulos.
- Também foram encontradas inconsistências de padronização em variáveis categóricas. O campo `distrito_sanitario`, que conceitualmente possui 8 categorias, apresentou **19 representações textuais**, enquanto `tipo_equipe`, esperado em 3 categorias (`eSF`, `eSB` e `eMulti`), apresentou **12 representações** devido a diferenças de capitalização, abreviação, hífen e espaços.
- Os campos `codigo_cnes` e `codigo_ine` foram interpretados como numéricos, embora sejam identificadores e devam ser tratados conceitualmente como texto.

## 2. Resultados dos Ciclos

- A tabela `resultados_ciclos` possui **1.392 registros** e representa os resultados de uma equipe em determinado ciclo.
- Foram identificados **12 pares repetidos da chave esperada `codigo_ine + ciclo`**. Entre esses pares, há duplicações aparentemente idênticas e casos com divergência entre os registros.
- O campo `ciclo` apresentou **43 representações textuais diferentes**, embora existam apenas 11 ciclos conceituais no período analisado. Foram encontrados **96 registros fora do formato esperado `AAAA.Q`**.
- A variável `eixo2_total` foi interpretada como texto. Foram identificados valores utilizando vírgula como separador decimal, além de **172 participantes com valor ausente nessa variável**.
- As **58 equipes desclassificadas** apresentam ausência de pontuação, comportamento compatível com a regra de que desclassificação não deve ser interpretada como pontuação zero.
- Na validação da taxa de resposta do Bloco A, foram encontrados **7 registros fora do intervalo de 0 a 1** e **4 registros inconsistentes com a regra de validade mínima de 30%**. Também foi observada diferença na representação de Bloco A inválido, com registros utilizando tanto `0` quanto `NaN`.
- Foram identificadas ainda **9 pontuações finais fora do intervalo de 0 a 1.000**, **8 classificações incompatíveis com a pontuação registrada** e **7 registros em que a pontuação final não corresponde à soma dos três eixos**.

## 3. Indicadores do Eixo 3

- A tabela `eixo3_indicadores` possui **13.294 registros**, sem valores ausentes e sem duplicidades na chave composta `codigo_ine + ciclo + codigo_indicador`.
- Foram encontrados os **11 ciclos esperados** e os **10 indicadores (`IND01` a `IND10`)**.
- A variável `percentual_alcancado` apresentou **380 registros acima de 1**. A análise conjunta com `peso_maximo` e `pontos_obtidos` demonstrou que esses registros utilizam escala percentual de 0 a 100, enquanto os demais utilizam escala de 0 a 1, caracterizando uma **mistura de escalas**.
- Após considerar essa diferença de representação, os **13.294 registros apresentaram cálculo de `pontos_obtidos` compatível com `peso_maximo × percentual_alcancado`**.
- A aplicabilidade dos indicadores varia conforme o grupo da equipe. Os pesos aplicáveis totalizaram **600 pontos para todos os quatro grupos**, e todas as **1.363 combinações equipe × ciclo** apresentaram soma de pesos igual a 600.
- Não foram encontrados INEs sem correspondência na tabela `equipes`.

## 4. NPS por Atributo

- A tabela `nps_atributos` possui **6.815 registros**, sem duplicidades na chave composta `codigo_ine + ciclo + codigo_atributo`.
- Foram encontrados os **5 atributos esperados (`ATR1` a `ATR5`)**, presentes nas **1.363 combinações equipe × ciclo**.
- Os valores preenchidos de NPS respeitam o intervalo esperado de **-100 a 100**, e não foram encontrados valores negativos em `n_respondentes`.
- Foram identificados **260 registros (3,82%) com NPS ausente mesmo havendo respondentes registrados**. Nesses casos, o número de respondentes varia de 3 a 127, indicando a necessidade de investigação antes da definição do tratamento.
- Não foram identificados INEs sem correspondência na tabela `equipes`.

## 5. Planos de Ação

- A tabela `planos_acao` possui **420 registros**, sem duplicidades em `id_plano`.
- As três dimensões avaliadas respeitam o intervalo esperado de 1 a 5, e `nivel_maturidade` respeita o intervalo de 1 a 3.
- A correspondência entre nível e rótulo de maturidade foi integralmente respeitada: nível 1 corresponde a `Reformulação`, nível 2 a `Ajustes` e nível 3 a `Aprovado`.
- Foram identificados **10 registros (2,38%) com `texto_plano` ausente**.
- Os **128 INEs distintos** presentes na tabela possuem correspondência em `equipes`, e todas as **420 combinações equipe × ciclo** possuem correspondência nos dados do Eixo 3.

## Conclusão do Entendimento dos Dados

O perfilamento demonstrou que a base apresenta estrutura suficiente para prosseguir para as etapas de preparação e análise, mas contém problemas de qualidade que precisam ser tratados de forma controlada.

Os principais pontos identificados foram:

- inconsistências de padronização em categorias e ciclos;
- tipos de dados inadequados para alguns identificadores e variáveis numéricas;
- valores ausentes que possuem significados diferentes conforme a regra de negócio;
- duplicidades na chave esperada de `resultados_ciclos`;
- valores fora de domínio em taxa de resposta e pontuação final;
- inconsistências entre pontuação, classificação e soma dos eixos;
- mistura de escalas em `percentual_alcancado`;
- NPS ausente mesmo na presença de respondentes;
- planos de ação sem conteúdo textual.

Por outro lado, também foram validadas regras importantes da estrutura da base, como a unicidade das principais chaves, a integridade referencial entre a maior parte das tabelas, a aplicação dos indicadores por grupo, a soma de 600 pontos no Eixo 3 e a correspondência entre nível e rótulo de maturidade dos planos de ação.
Nesta etapa, os problemas foram **identificados e documentados, sem alteração dos dados originais**. As decisões de correção, padronização, conversão ou manutenção dos valores serão definidas na etapa de Preparação dos Dados, considerando as regras de negócio e evitando tratamentos automáticos que possam alterar o significado dos registros.